# Construction PPE Detection

Цель проекта: построить модель computer vision, которая находит на изображениях строительные/производственные средства индивидуальной защиты: каски, жилеты и головы без каски.

Формат задачи: **object detection**. Это значит, что модель не только определяет класс объекта, но и рисует вокруг него прямоугольник — bounding box.

## План проекта

1. Проверить структуру датасета.
2. Изучить разметку и баланс классов.
3. Визуально проверить bounding boxes на примерах.
4. Обучить YOLO-модель.
5. Оценить качество модели.
6. Сделать скрипт предсказания.
7. Сделать API и web-интерфейс.
8. Упаковать проект в Docker.

## Импорт библиотек

`pathlib` помогает работать с путями к файлам.

`pandas` нужен для табличной статистики по разметке.

`matplotlib` нужен для графиков.

`Pillow` нужен для открытия изображений и рисования рамок.

`yaml` читает файл `data.yaml`, где описаны пути к датасету и классы для YOLO.

In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from PIL import Image, ImageDraw, ImageFont

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_YAML = PROJECT_ROOT / "data" / "data.yaml"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

PROJECT_ROOT

## Читаем `data.yaml`

`data.yaml` — это главный конфиг для YOLO. В нём написано:

- где лежит датасет;
- где лежат `train`, `val`, `test` изображения;
- какие id классов соответствуют каким названиям.

In [ ]:
with DATA_YAML.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

dataset_root = PROJECT_ROOT / config["path"]

config, dataset_root

## Проверяем количество изображений и label-файлов

Для YOLO обычно каждому изображению соответствует `.txt` файл с тем же именем.

Пример:

`000009.jpg` -> `000009.txt`

In [ ]:
rows = []

for split in ["train", "val", "test"]:
    images_dir = dataset_root / config[split]
    labels_dir = dataset_root / "labels" / split

    image_files = [p for p in images_dir.glob("*") if p.suffix.lower() in IMAGE_EXTENSIONS]
    label_files = list(labels_dir.glob("*.txt"))

    rows.append({
        "split": split,
        "images": len(image_files),
        "labels": len(label_files),
    })

files_df = pd.DataFrame(rows)
files_df

## Формат YOLO-разметки

Одна строка в label-файле выглядит так:

`class_id x_center y_center width height`

Координаты нормализованы: значения лежат от `0` до `1`, а не в пикселях. Это удобно, потому что модель может работать с изображениями разного размера.

In [ ]:
sample_label = next((dataset_root / "labels" / "train").glob("*.txt"))

print(sample_label.name)
print(sample_label.read_text(encoding="utf-8")[:500])

## Собираем статистику по объектам

Теперь превращаем все `.txt` файлы в одну таблицу. Каждая строка таблицы — один найденный объект в разметке.

In [ ]:
records = []

for split in ["train", "val", "test"]:
    labels_dir = dataset_root / "labels" / split

    for label_path in labels_dir.glob("*.txt"):
        for line in label_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue

            class_id, x_center, y_center, width, height = line.split()
            class_id = int(class_id)

            records.append({
                "split": split,
                "image_id": label_path.stem,
                "class_id": class_id,
                "class_name": config["names"][class_id],
                "x_center": float(x_center),
                "y_center": float(y_center),
                "width": float(width),
                "height": float(height),
                "box_area": float(width) * float(height),
            })

annotations_df = pd.DataFrame(records)
annotations_df.head()

In [ ]:
class_distribution = (
    annotations_df
    .groupby(["split", "class_name"])
    .size()
    .reset_index(name="objects")
    .sort_values(["split", "objects"], ascending=[True, False])
)

class_distribution

In [ ]:
pivot = class_distribution.pivot(index="class_name", columns="split", values="objects").fillna(0)

ax = pivot.plot(kind="bar", figsize=(9, 5))
ax.set_title("Class distribution by split")
ax.set_xlabel("Class")
ax.set_ylabel("Objects")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Первые выводы

- Датасет большой: больше 22 тысяч изображений.
- Разметка полная: для каждого изображения есть label-файл.
- Классы несбалансированы: `head` встречается намного чаще, чем `vest`.
- При оценке модели нужно отдельно смотреть качество по каждому классу, особенно по `vest`.

## Визуальная проверка разметки

Перед обучением важно глазами проверить несколько изображений. Если рамки нарисованы неправильно, модель будет учиться на ошибочных данных.

In [ ]:
def yolo_to_xyxy(x_center, y_center, width, height, image_width, image_height):
    box_width = width * image_width
    box_height = height * image_height
    x1 = int((x_center * image_width) - box_width / 2)
    y1 = int((y_center * image_height) - box_height / 2)
    x2 = int((x_center * image_width) + box_width / 2)
    y2 = int((y_center * image_height) + box_height / 2)
    return x1, y1, x2, y2


def draw_yolo_boxes(image_path, label_path):
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    image_width, image_height = image.size

    colors = {
        0: "green",
        1: "orange",
        2: "red",
        3: "royalblue",
    }

    for line in label_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue

        class_id, x_center, y_center, width, height = line.split()
        class_id = int(class_id)
        x1, y1, x2, y2 = yolo_to_xyxy(
            float(x_center), float(y_center), float(width), float(height), image_width, image_height
        )

        label = config["names"][class_id]
        color = colors[class_id]
        draw.rectangle((x1, y1, x2, y2), outline=color, width=3)
        draw.text((x1, max(y1 - 14, 0)), label, fill=color, font=font)

    return image

In [ ]:
train_images_dir = dataset_root / config["train"]
train_labels_dir = dataset_root / "labels" / "train"

sample_images = list(train_images_dir.glob("*.jpg"))[:4]

fig, axes = plt.subplots(2, 2, figsize=(12, 12))

for ax, image_path in zip(axes.ravel(), sample_images):
    label_path = train_labels_dir / f"{image_path.stem}.txt"
    image = draw_yolo_boxes(image_path, label_path)
    ax.imshow(image)
    ax.set_title(image_path.name)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Следующий шаг

После EDA переходим к обучению YOLO. Для этого установим библиотеку `ultralytics` и создадим `src/train.py`.

Сначала запустим короткое обучение на небольшом числе эпох, чтобы проверить весь pipeline. Потом уже будем улучшать качество.